In [2]:
import subprocess
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import sys

sys.path.append("../..")
from src import *


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/vmorelli-iit.local/miniconda3/envs/texture-anything/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/vmorelli-iit.local/miniconda3/envs/texture-anything/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/vmorelli-iit.local/miniconda3/envs/texture-anything/lib/python3.

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Unable to initialise audio


ImportError: numpy.core.multiarray failed to import

/home/vmorelli-iit.local/miniconda3/envs/texture-anything/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
OUTPUT_DIR = Path("dataset")
(OUTPUT_DIR/"diffuse").mkdir(exist_ok=True, parents=True)
(OUTPUT_DIR/"uv").mkdir(exist_ok=True)
TESTSET_DIR = Path("../dataset/test")
testset = pd.read_json("../dataset/test/metadata.jsonl", orient="records", lines=True)

In [4]:
ckpt = "stabilityai/stable-diffusion-xl-base-1.0"
controlnet = "trainings/sdxl_16bs_-5lr_2k_inv-000005.safetensors"
# controlnet = "checkpoints/bdsqlsz_controlllite_xl_mlsd_V2.safetensors"
steps = 30
out_dir = Path("outputs") / controlnet.split("/")[-1]
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
for _, sample in tqdm(list(testset.iterrows())):
    cmd = [
        "python",
        "sdxl_gen_img.py",
        "--ckpt",
        ckpt,
        "--control_net_lllite_models",
        controlnet,
        "--guide_image_path",
        str((TESTSET_DIR / sample.uv_file_name).resolve()),
        "--prompt",
        sample.caption,
        "--outdir",
        str(out_dir),
        "--use_original_file_name",
        "--W",
        "1024",
        "--H",
        "1024",
        "--bf16",
        "--batch_size",
        "1",
        "--steps",
        str(steps),
    ]
    subprocess.run(cmd, check=True, capture_output=False)

In [6]:
out_dir = Path("outputs") / "sdxl_16bs_-5lr_2k_inv-000010.safetensors"
outputs = sorted(out_dir.glob("*.png"), key=lambda f:f.stem.split("_")[1])
for uid, file in zip(testset.uv_file_name, outputs):
    file.rename(file.with_stem(Path(uid).stem))